In [6]:
import asyncio
from neo4j import GraphDatabase
from neo4j_graphrag.embeddings import AzureOpenAIEmbeddings
from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline
from neo4j_graphrag.llm import AzureOpenAILLM
from neo4j_graphrag.experimental.components.text_splitters.fixed_size_splitter import FixedSizeSplitter
from neo4j_graphrag.retrievers import VectorRetriever
from neo4j_graphrag.generation.graphrag import GraphRAG
from neo4j_graphrag.indexes import create_vector_index
from neo4j_graphrag.generation.prompts import RagTemplate
from IPython.display import Markdown
from config import azure_config
import os

NEO4J_URI = "neo4j://localhost:7687"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "password"

In [2]:
neo4j_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

In [3]:
neo4j_driver.verify_connectivity()

In [4]:
llm = AzureOpenAILLM(
    model_name=azure_config.model_name,
    model_params={"temperature": 0},
    api_key=azure_config.api_key,
    azure_endpoint=azure_config.endpoint,
    api_version=azure_config.api_version,
    azure_deployment="gpt-4o"    
)

embedder = AzureOpenAIEmbeddings(
    model=azure_config.embedding_model_name,
    api_key=azure_config.api_key,
    azure_endpoint=azure_config.endpoint,
    api_version=azure_config.api_version,
    # azure_deployment="text-embedding-3-small"
)

In [ ]:
# Get a list of all PDF file paths in the 'pdf' folder
pdf_folder = "pdfs"
pdf_files = [os.path.join(pdf_folder, file) for file in os.listdir(pdf_folder) if file.endswith(".pdf")]

print(pdf_files)

['pdf/2023Q1_alphabet_earnings_release.pdf', 'pdf/Meta-Reports-First-Quarter-2023-Results-2023.pdf', 'pdf/TSLA-Q1-2023-Update.pdf', 'pdf/NVIDIA Q1FY23-CFO-Commentary.pdf']


In [35]:
from markitdown import MarkItDown

md = MarkItDown()
pdf_content = md.convert("Graph_Databases_for_Beginners.pdf").text_content
# print(result.text_content)

In [ ]:
prompt_template = '''
You are a medical researcher tasks with extracting information from papers 
and structuring it in a property graph to inform further medical and research Q&A.

Extract the entities (nodes) and specify their type from the following Input text.
Also extract the relationships between these nodes. the relationship direction goes from the start node to the end node. 


Return result as JSON using the following format:
{{"nodes": [ {{"id": "0", "label": "the type of entity", "properties": {{"name": "name of entity" }} }}],
  "relationships": [{{"type": "TYPE_OF_RELATIONSHIP", "start_node_id": "0", "end_node_id": "1", "properties": {{"details": "Description of the relationship"}} }}] }}

- Use only the information from the Input text. Do not add any additional information.  
- If the input text is empty, return empty Json. 
- Make sure to create as many nodes and relationships as needed to offer rich medical context for further research.
- An AI knowledge assistant must be able to read this graph and immediately understand the context to inform detailed research questions. 
- Multiple documents will be ingested from different sources and we are using this property graph to connect information, so make sure entity types are fairly general. 

Use only fhe following nodes and relationships (if provided):
{schema}

Assign a unique ID (string) to each node, and reuse it to define relationships.
Do respect the source and target node types for relationship and
the relationship direction.

Do not return any additional information other than the JSON in it.

Examples:
{examples}

Input text:

{text}
'''

# List of Entities
entities = [
    "Company",
    "Quarter",
    "Revenue",
    "Net Income",
    "Expenses",
    "Assets",
    "Liabilities",
    "Shareholders Equity",
    "Earnings Per Share (EPS)",
    "Market Capitalization",
    "Dividend",
    "Operating Income",
    "Cash Flow",
    "Financial Ratios",
    "Management Commentary",
    "Auditor's Report",
    "Stakeholders",
    "Industry",
    "Products/Services",
]
# List of Relationships
relations = [
    "reports",
    "generates",
    "includes",
    "has",
    "is part of",
    "compares to",
    "influences",
    "affects",
    "is reported by",
    "is evaluated by",
    "belongs to",
    "is a type of",
    "is measured by",
    "divides",
    "increases",
    "decreases",
]

kg_builder_pdf = SimpleKGPipeline(
    llm=llm,
    driver=neo4j_driver,
    embedder=embedder,
    entities=entities,
    relations=relations,
    prompt_template=prompt_template,
    text_splitter=FixedSizeSplitter(chunk_size=500, chunk_overlap=100),
    on_error="IGNORE",
    from_pdf=True,
)

# await kg_builder_pdf.run_async(file_path="Graph_Databases_for_Beginners.pdf",)

# generate a async function to run the pipeline
async def run_pipeline(file_path):
    print(f"Processing : {file_path}")
    data = await kg_builder_pdf.run_async(file_path=file_path)
    print(f"Result: {data}")
    return data

async def run_in_chunks(pdf_files, chunk_size=5):
    results = []
    for i in range(0, len(pdf_files), chunk_size):
        chunk = pdf_files[i:i + chunk_size]
        list_of_tasks = [run_pipeline(file_path) for file_path in chunk]
        chunk_results = await asyncio.gather(*list_of_tasks)
        results.extend(chunk_results)
    return results

# Example usage with async function
result_async = await run_in_chunks(pdf_files)

Processing : pdf/2023Q1_alphabet_earnings_release.pdf

Processing : pdf/Meta-Reports-First-Quarter-2023-Results-2023.pdf

Processing : pdf/TSLA-Q1-2023-Update.pdf

Processing : pdf/NVIDIA Q1FY23-CFO-Commentary.pdf

Result: run_id='c80009d4-b6e5-4fbe-87e1-5676f2631231' result={'resolver': {'number_of_nodes_to_resolve': 336, 
'number_of_created_nodes': 250}}

Result: run_id='8904b9fc-402c-43b6-a3c0-ae4b71b8b00b' result={'resolver': {'number_of_nodes_to_resolve': 620, 
'number_of_created_nodes': 502}}

Result: run_id='c9dde3f0-2cc2-438c-83d8-0c49c99af770' result={'resolver': {'number_of_nodes_to_resolve': 1009, 
'number_of_created_nodes': 826}}

Result: run_id='2b1b4088-2b94-4716-8be5-cfd09358fff3' result={'resolver': {'number_of_nodes_to_resolve': 1568, 
'number_of_created_nodes': 1252}}

In [43]:
result_async

[PipelineResult(run_id='c9dde3f0-2cc2-438c-83d8-0c49c99af770', result={'resolver': {'number_of_nodes_to_resolve': 1009, 'number_of_created_nodes': 826}}),
 PipelineResult(run_id='c80009d4-b6e5-4fbe-87e1-5676f2631231', result={'resolver': {'number_of_nodes_to_resolve': 336, 'number_of_created_nodes': 250}}),
 PipelineResult(run_id='2b1b4088-2b94-4716-8be5-cfd09358fff3', result={'resolver': {'number_of_nodes_to_resolve': 1568, 'number_of_created_nodes': 1252}}),
 PipelineResult(run_id='8904b9fc-402c-43b6-a3c0-ae4b71b8b00b', result={'resolver': {'number_of_nodes_to_resolve': 620, 'number_of_created_nodes': 502}})]

In [ ]:
# Run the cyphere query to check the data:
# MATCH p=()-->() RETURN p LIMIT 1000; 

In [ ]:
INDEX_NAME = "vector-index-name"

# Create the index
create_vector_index(
    neo4j_driver,
    INDEX_NAME,
    label="Chunk",
    embedding_property="embedding",
    dimensions=1536,
    # similarity_fn="euclidean",
    similarity_fn="cosine",
)

In [7]:
INDEX_NAME = "vector-index-name"
class MyRagTemplate(RagTemplate):
    DEFAULT_SYSTEM_INSTRUCTIONS = """
    Answer the user question using the provided context.
    if the context has table and its relevant to the question, 
    then use the table to answer the question in markdown format.
    Always provide a useful insigh after the answer, and mention the source of the information.
    Never say that your answer is based on the context,
    """


# Initialize the retriever
retriever = VectorRetriever(neo4j_driver, INDEX_NAME, embedder)

# Instantiate the RAG pipeline
rag = GraphRAG(retriever=retriever, llm=llm, prompt_template=MyRagTemplate())

In [72]:
# query_text = "Extract useful financial tables for Alphabet and Tesla, ensuring that both tables have the same columns and cover the same period. Then, generate 5 insightful comparative questions about both companies and provide answers to those questions"
# query_text = "How has a financial performance and strategic decisions, including the timing shift of employee stock-based compensation awards, the authorization of a $70 billion stock repurchase, and the impact of unallocated corporate costs, influenced its balance sheet and segment results for FY 2022 and Q1 2023, and what are the potential risks and uncertainties highlighted in their forward-looking statements?"
# query_text = "Show me the table Balance Sheet and Segment Results for FY 2022 and Q1 2023, and give the most powerful insight"
query_text = "Compare Q1-2023 revenues between Alphabet Inc and Tesla"
response = rag.search(query_text=query_text, retriever_config={"top_k": 3, }, return_context=True)
print(response.answer)

Based on the provided context, here are the key financial metrics for Tesla in Q1-2023:

- **Operating Margin**: 11.4%
- **GAAP Operating Income**: $2.7 billion
- **GAAP Net Income**: $2.5 billion
- **Non-GAAP Net Income**: $2.9 billion

Unfortunately, the context does not provide specific revenue figures for Tesla in Q1-2023, nor does it provide any 
financial data for Alphabet Inc. in Q1-2023. Therefore, I cannot directly compare the Q1-2023 revenues between 
Alphabet Inc. and Tesla.

**Insight:**
To make a comprehensive comparison of Q1-2023 revenues between Alphabet Inc. and Tesla, it would be necessary to 
obtain the specific revenue figures for both companies. This information is typically available in their respective
quarterly earnings reports.

**Source:**
The information provided is based on the context given, which includes Tesla's financial metrics and a brief 
mention of Alphabet Inc.'s Q1-2023 results.

In [73]:
from rich import print
print(f"Length:{len(response.retriever_result.items)}")

Length:3

In [74]:
print(list(map(lambda x: x.metadata['score'],response.retriever_result.items)))

[0.5953876972198486, 0.5739139318466187, 0.5478354692459106]

In [ ]:
print(f"Items:{response.retriever_result.items}")

Items:[RetrieverResultItem(content="{'id': '0da5799a-e6de-45a5-a611-9c89bde1d4ad', 'index': 39, 'text': 
'Q1-2019\\nQ2-2019\\nQ3-2019\\nQ4-2019\\nQ1-2020\\nQ2-2020\\nQ3-2020\\nQ4-2020\\nQ1-2021\\nQ2-2021\\nQ3-2021\\nQ4-2
021\\nQ1-2022\\nQ2-2022\\nQ3-2022\\nQ4-2022\\nQ1-2023\\nTesla Auto Industry S&P 
500\\n-20%\\n-10%\\n0%\\n10%\\n20%\\n30%\\n40%\\n50%\\n60%\\n70%\\n80%\\n90%\\nQ1-2019\\nQ2-2019\\nQ3-2019\\nQ4-201
9\\nQ1-2020\\nQ2-2020\\nQ3-2020\\nQ4-2020\\nQ1-2021\\nQ2-2021\\nQ3-2021\\nQ4-2021\\nQ1-2022\\nQ2-2022\\nQ3-2022\\nQ
4-2022\\nQ1-2023\\nTesla Auto Industry S&P 500\\nK E Y M E T R I C S T R A I L I N G 1 2 M O N T H S ( T T M 
)\\n(Unaudited)\\n21\\nYoY Revenue Growth Operating Margin\\nSource: ', 'embedding': None}", metadata={'score': 
0.5953876972198486, 'nodeLabels': ['__KGBuilder__', 'Chunk'], 'id': 
'4:91ec46b8-5f22-418d-9f9a-dba7549ade07:1140'}), RetrieverResultItem(content="{'id': 
'8d1d1c6c-dc8d-480b-8a10-97c74f193b75', 'index': 1, 'text': 'and investments.\\nProfitability 11.4% operating 
margin in Q1\\n$2.7B GAAP operating income in Q1\\n$2.5B GAAP net income in Q1\\n$2.9B non-GAAP net income 1 in 
Q1\\nIn the current macroeconomic environment, we see this year as a unique \\nopportunity for Tesla. As many 
carmakers are working through challenges with the \\nunit economics of their EV programs, we aim to leverage our 
position as a cost \\nleader. We are focused on rapidly growing production, investments in autonomy \\nand vehicle 
software, and ', 'embedding': None}", metadata={'score': 0.5739139318466187, 'nodeLabels': ['__KGBuilder__', 
'Chunk'], 'id': '4:91ec46b8-5f22-418d-9f9a-dba7549ade07:1032'}), RetrieverResultItem(content="{'id': 
'da95aada-606d-40f5-913c-8f67dedea1e9', 'index': 0, 'text': 'Alphabet Announces First Quarter 2023 
Results\\nMOUNTAIN VIEW, Calif. – April 25, 2023  – Alphabet Inc. (NASDAQ: GOOG, GOOGL) today announced financial 
\\nresults for the quarter ended March 31, 2023.\\nSundar Pichai, CEO of Alphabet and Google, said: “We are pleased
with our business performance in the first \\nquarter, with Search performing well and momentum in Cloud. We 
introduced important product updates anchored \\nin deep computer science and AI. Our North Star is providing the 
most helpful ', 'embedding': None}", metadata={'score': 0.5478354692459106, 'nodeLabels': ['__KGBuilder__', 
'Chunk'], 'id': '4:91ec46b8-5f22-418d-9f9a-dba7549ade07:114'})]

In [76]:
k = 10
response = rag.search(query_text=query_text, retriever_config={"top_k": k, }, return_context=True)
Markdown(response.answer)

Based on the provided context, here is the comparison of Q1-2023 revenues between Alphabet Inc. and Tesla:

### Q1-2023 Revenues

| Company       | Q1-2023 Revenues |
|---------------|------------------|
| Alphabet Inc. | $69.8 billion    |
| Tesla         | $23.329 billion  |

**Insight:**
Alphabet Inc.'s revenues in Q1-2023 were significantly higher than Tesla's, with Alphabet generating nearly three times the revenue of Tesla during this period. This highlights the substantial difference in scale between the two companies, with Alphabet's diverse portfolio in technology and advertising contributing to its higher revenue.

**Source:**
- Alphabet Inc. financial results for Q1-2023
- Tesla financial summary for Q1-2023

In [77]:
print(f"Items:{response.retriever_result.items}")

Items:[RetrieverResultItem(content="{'id': '0da5799a-e6de-45a5-a611-9c89bde1d4ad', 'index': 39, 'text': 
'Q1-2019\\nQ2-2019\\nQ3-2019\\nQ4-2019\\nQ1-2020\\nQ2-2020\\nQ3-2020\\nQ4-2020\\nQ1-2021\\nQ2-2021\\nQ3-2021\\nQ4-2
021\\nQ1-2022\\nQ2-2022\\nQ3-2022\\nQ4-2022\\nQ1-2023\\nTesla Auto Industry S&P 
500\\n-20%\\n-10%\\n0%\\n10%\\n20%\\n30%\\n40%\\n50%\\n60%\\n70%\\n80%\\n90%\\nQ1-2019\\nQ2-2019\\nQ3-2019\\nQ4-201
9\\nQ1-2020\\nQ2-2020\\nQ3-2020\\nQ4-2020\\nQ1-2021\\nQ2-2021\\nQ3-2021\\nQ4-2021\\nQ1-2022\\nQ2-2022\\nQ3-2022\\nQ
4-2022\\nQ1-2023\\nTesla Auto Industry S&P 500\\nK E Y M E T R I C S T R A I L I N G 1 2 M O N T H S ( T T M 
)\\n(Unaudited)\\n21\\nYoY Revenue Growth Operating Margin\\nSource: ', 'embedding': None}", metadata={'score': 
0.5953876972198486, 'nodeLabels': ['__KGBuilder__', 'Chunk'], 'id': 
'4:91ec46b8-5f22-418d-9f9a-dba7549ade07:1140'}), RetrieverResultItem(content="{'id': 
'8d1d1c6c-dc8d-480b-8a10-97c74f193b75', 'index': 1, 'text': 'and investments.\\nProfitability 11.4% operating 
margin in Q1\\n$2.7B GAAP operating income in Q1\\n$2.5B GAAP net income in Q1\\n$2.9B non-GAAP net income 1 in 
Q1\\nIn the current macroeconomic environment, we see this year as a unique \\nopportunity for Tesla. As many 
carmakers are working through challenges with the \\nunit economics of their EV programs, we aim to leverage our 
position as a cost \\nleader. We are focused on rapidly growing production, investments in autonomy \\nand vehicle 
software, and ', 'embedding': None}", metadata={'score': 0.5739139318466187, 'nodeLabels': ['__KGBuilder__', 
'Chunk'], 'id': '4:91ec46b8-5f22-418d-9f9a-dba7549ade07:1032'}), RetrieverResultItem(content="{'id': 
'da95aada-606d-40f5-913c-8f67dedea1e9', 'index': 0, 'text': 'Alphabet Announces First Quarter 2023 
Results\\nMOUNTAIN VIEW, Calif. – April 25, 2023  – Alphabet Inc. (NASDAQ: GOOG, GOOGL) today announced financial 
\\nresults for the quarter ended March 31, 2023.\\nSundar Pichai, CEO of Alphabet and Google, said: “We are pleased
with our business performance in the first \\nquarter, with Search performing well and momentum in Cloud. We 
introduced important product updates anchored \\nin deep computer science and AI. Our North Star is providing the 
most helpful ', 'embedding': None}", metadata={'score': 0.5478354692459106, 'nodeLabels': ['__KGBuilder__', 
'Chunk'], 'id': '4:91ec46b8-5f22-418d-9f9a-dba7549ade07:114'}), RetrieverResultItem(content="{'id': 
'805e5e2a-9c17-41c6-a4e8-f695cf0797d1', 'index': 36, 'text': 'Class B 883, Class C 5,896) shares issued and 
outstanding\\n 68,184  70,269 \\nAccumulated other comprehensive income (loss)  (7,603)  (6,000) \\nRetained 
earnings  195,563  196,625 \\nTotal stockholders’ equity  256,144  260,894 \\nTotal liabilities and stockholders’ 
equity $ 365,264 $ 369,491 \\n5\\nAlphabet Inc.\\nCONSOLIDATED STATEMENTS OF INCOME\\n(In millions, except per 
share amounts, unaudited)\\nQuarter Ended March 31,\\n2022 2023\\nRevenues $ 68,011 $ 69,787 \\nCosts and 
expenses:\\nCost of revenues  29,599  ', 'embedding': None}", metadata={'score': 0.5414543747901917, 'nodeLabels': 
['__KGBuilder__', 'Chunk'], 'id': '4:91ec46b8-5f22-418d-9f9a-dba7549ade07:500'}), 
RetrieverResultItem(content="{'id': '1fea8609-3c9a-4e9a-8e78-8273b58b67ea', 'index': 67, 'text': ' 21 186 83 69 115
223 292 346 205 305 276 261 \\nDepreciation, amortization and impairment 530 577 553 567 584 618 621 681 761 848 
880 922 956 989 1,046 \\nStock-based compensation expense 199 281 211 347 543 633 614 474 475 558 418 361 362 419 
418 \\nAdjusted EBITDA (non-GAAP) 1,083 1,175 951 1,209 1,807 1,850 1,841 2,487 3,203 4,090 5,023 3,791 4,968 5,404
4,267 \\n27\\nA D D I T I O N A L I N F O R M A T I O N\\nWEBCAST INFORMATION\\nTesla will provide a live webcast 
of its first quarter 2023 financial ', 'embedding': None}", metadata={'score': 0.5380926132202148, 'nodeLabels': 
['__KGBuilder__', 'Chunk'], 'id': '4:91ec46b8-5f22-418d-9f9a-dba7549ade07:721'}), 
RetrieverResultItem(

In [90]:
query_text = "Tell me about Nvidia free cash flow and its partners’ products, and then tell me about Meta Platforms, Inc. with its financial results"
response = rag.search(query_text=query_text, retriever_config=
                      {
                          "top_k": 10, 
                          },
                      return_context=True)
Markdown(response.answer)

### NVIDIA Free Cash Flow and Partners' Products

NVIDIA reported a free cash flow of $6.911 billion for the first quarter of fiscal 2023. This figure is derived after accounting for purchases of property and equipment, net, which amounted to $6.823 billion, and principal payments on finance leases, which were $264 million.

NVIDIA's platforms address four large markets where their expertise is critical: Gaming, Data Center, Professional Visualization, and Automotive. They deliver unique value through their products, which include interconnects, software, algorithms, systems, and services.

### Meta Platforms, Inc. Financial Results

Meta Platforms, Inc. reported the following financial results for the first quarter of 2023:

- **Revenue**: $28.645 billion, up from $27.908 billion in the same period of 2022.
- **Net Income**: $5.709 billion, down from $7.465 billion in the same period of 2022.
- **Free Cash Flow**: $6.911 billion, down from $8.528 billion in the same period of 2022.

#### Condensed Consolidated Statements of Income (In millions, except per share amounts)

| Three Months Ended March 31, | 2023     | 2022     |
|------------------------------|----------|----------|
| Revenue                      | $28,645  | $27,908  |
| Cost of Revenue              | $6,108   | $6,005   |
| Research and Development     | $9,381   | $7,707   |
| Marketing and Sales          | $3,044   | $3,312   |
| General and Administrative   | $2,885   | $2,360   |

Meta's financial performance indicates a slight increase in revenue but a decrease in net income and free cash flow compared to the previous year. This suggests that while the company is growing its top line, it is also facing higher costs and expenses.

**Insight**: Both NVIDIA and Meta Platforms, Inc. are showing strong revenue figures, but Meta is experiencing a decline in net income and free cash flow, indicating potential challenges in managing costs or investments. NVIDIA's significant free cash flow highlights its strong financial health and ability to invest in future growth.

**Source**: Meta Platforms, Inc. financial results and NVIDIA financial data.

In [91]:
print(f"Items:{response.retriever_result.items}")

Items:[RetrieverResultItem(content="{'id': '15d03eb0-9fa6-4b2b-82d3-6c15b9ab371b', 'index': 29, 'text': 'er diluted
share, and free cash flow. In order for NVIDIA’s investors to be better able to \\ncompare its current results with
those of previous periods, the company has shown a reconciliation of GAAP \\nto non-GAAP financial measures. These 
reconciliations adjust the related GAAP financial measures to exclude \\nacquisition termination costs, stock-based
compensation expense, acquisition-related and other costs, IP-\\nrelated costs, legal settlement costs, gains and 
losses from non-affiliated ', 'embedding': None}", metadata={'score': 0.5448687672615051, 'nodeLabels': 
['__KGBuilder__', 'Chunk'], 'id': '4:91ec46b8-5f22-418d-9f9a-dba7549ade07:397'}), 
RetrieverResultItem(content="{'id': 'd6c3b3df-d5ae-4008-bcd2-b69648c373b0', 'index': 4, 'text': ' interconnects, 
software, algorithms, systems, and \\nservices to deliver unique value. Our platforms address four large markets 
where our expertise is critical: \\nGaming, Data Center, Professional Visualization, and 
Automotive.\\nRevenue\\nRevenue was a record $8.29 billion, up 46% from a year ago and up 8% sequentially, with 
record revenue \\nachieved in Gaming and Data Center. \\nGaming revenue was up 31% from a year ago and up 6% 
sequentially. The year-on-year increase reflects \\nhigher sales of GeForce', 'embedding': None}", 
metadata={'score': 0.5250647068023682, 'nodeLabels': ['__KGBuilder__', 'Chunk'], 'id': 
'4:91ec46b8-5f22-418d-9f9a-dba7549ade07:372'}), RetrieverResultItem(content='{\'id\': 
\'2f25d134-2438-4600-b8bb-a9845965966e\', \'index\': 40, \'text\': ",241 64,799\\nTotal stockholders\' equity 
124,795 125,713\\nTotal liabilities and stockholders\' 
equity\\n$\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0184,491 
$\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0 185,727\\n \\n \\n \\nMETA PLATFORMS, 
INC.\\nCONDENSED CONSOLIDATED STATEMENTS OF CASH FLOWS\\n(In millions)\\n(Unaudited)\\n Three Months Ended March 
31,\\n 2023 2022\\nCash \\x00ows from operating activities    \\nNet income 
$\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa05,709 
$\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0 7,465\\nAdjustments to 
reconcile net income to net cash provided by operating activities:   \\nDepreciation and ", \'embedding\': None}', 
metadata={'score': 0.5203520059585571, 'nodeLabels': ['__KGBuilder__', 'Chunk'], 'id': 
'4:91ec46b8-5f22-418d-9f9a-dba7549ade07:17'}), RetrieverResultItem(content="{'id': 
'07796d97-90fd-453a-93c8-f5a5ce0c118f', 'index': 46, 'text': 'equivalents, and restricted 
cash\\n$\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa012,420 
$\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0 15,353\\n \\n \\n \\nMETA PLATFORMS, 
INC.\\nCONDENSED CONSOLIDATED STATEMENTS OF CASH FLOWS\\n(In millions)\\n(Unaudited)\\n Three Months Ended March 
31,\\n 2023 2022\\nSupplemental cash \\x00ow data    \\nCash paid for income taxes, net $\\xa0 \\xa0 \\xa0 \\xa0 
\\xa0 \\xa0 \\xa0 \\xa0 \\xa0 \\xa0 \\xa0 405 $ 502\\nCash paid for interest, net of amounts capitalized$\\xa0 
\\xa0 \\xa0 \\xa0 \\xa0 \\xa0 \\xa0 \\xa0 \\xa0 \\xa0 \\xa0 182 $ —\\nNon-cash investing and \\x00nancing 
activities:      \\nProperty and equipment in accounts payable and ', 'embedding': None}", metadata={'score': 
0.5174223184585571, 'nodeLabels': ['__KGBuilder__', 'Chunk'], 'id': '4:91ec46b8-5f22-418d-9f9a-dba7549ade07:23'}), 
RetrieverResultItem(content="{'id': '2308f701-20ff-469b-9686-63687fade6a5', 'index': 53, 'text': 
'14,076\\nPurchases of property and equipment, net (6,823) (5,315)\\nPrincipal payments on \\x00nance leases (264) 
(233)\\n9\\nFree cash \\x00ow  $\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0 6,911 
$\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0\\xa0 8,528\\n\\xa0\\n\\xa0\\nView o

In [94]:
query_text = "When will Tesla's first quarter 2023 financial results conference call be webcast, and where can it be accessed? How long will the replay be available?"
response = rag.search(query_text=query_text, retriever_config=
                      {
                          "top_k": 3, 
                          },
                      return_context=True)
Markdown(response.answer)

Tesla's first quarter 2023 financial results conference call will be webcast live beginning at 4:30 p.m. CT on April 19, 2023. The webcast can be accessed at [ir.tesla.com](http://ir.tesla.com). The replay of the webcast will be available for approximately one year thereafter.

Insight: Keeping track of such webcasts can provide valuable insights into Tesla's financial performance and strategic direction. This information can be particularly useful for investors and analysts.

Source: Tesla's webcast information.

In [95]:
print(f"Items:{response.retriever_result.items}")

Items:[RetrieverResultItem(content="{'id': '53651d56-00e3-438e-9e2d-768427d096fd', 'index': 68, 'text': ' I O 
N\\nWEBCAST INFORMATION\\nTesla will provide a live webcast of its first quarter 2023 financial results conference 
call beginning at 4:30 p.m. CT on April 19, 2023 at ir.tesla.com. This webcast will also be available for replay 
for approximately one year \\nthereafter.\\nCERTAIN TERMS\\nWhen used in this update, certain terms have the 
following meanings. Our vehicle deliveries include only vehicles that have been transferred to end customers with 
all paperwork correctly completed. Our energy product\\n', 'embedding': None}", metadata={'score': 
0.6626668572425842, 'nodeLabels': ['__KGBuilder__', 'Chunk'], 'id': '4:91ec46b8-5f22-418d-9f9a-dba7549ade07:740'}),
RetrieverResultItem(content="{'id': '1fea8609-3c9a-4e9a-8e78-8273b58b67ea', 'index': 67, 'text': ' 21 186 83 69 115
223 292 346 205 305 276 261 \\nDepreciation, amortization and impairment 530 577 553 567 584 618 621 681 761 848 
880 922 956 989 1,046 \\nStock-based compensation expense 199 281 211 347 543 633 614 474 475 558 418 361 362 419 
418 \\nAdjusted EBITDA (non-GAAP) 1,083 1,175 951 1,209 1,807 1,850 1,841 2,487 3,203 4,090 5,023 3,791 4,968 5,404
4,267 \\n27\\nA D D I T I O N A L I N F O R M A T I O N\\nWEBCAST INFORMATION\\nTesla will provide a live webcast 
of its first quarter 2023 financial ', 'embedding': None}", metadata={'score': 0.6105561256408691, 'nodeLabels': 
['__KGBuilder__', 'Chunk'], 'id': '4:91ec46b8-5f22-418d-9f9a-dba7549ade07:721'}), 
RetrieverResultItem(content="{'id': '685a7fdf-d295-494e-80d4-87dd70af6b36', 'index': 19, 'text': ' Class C shares. 
The repurchases are expected to be executed from time to \\ntime, subject to general business and market conditions
and other investment opportunities, through open market \\npurchases or privately negotiated transactions, 
including through Rule 10b5-1 plans.\\nWebcast and conference call information\\nA live audio webcast of our first 
quarter  2023 earnings release call will be available on YouTube at https://\\nwww.youtube.com/watch?v=76CVRgZUfps.
The call begins today at 2:00 PM (PT) /', 'embedding': None}", metadata={'score': 0.5291087627410889, 'nodeLabels':
['__KGBuilder__', 'Chunk'], 'id': '4:91ec46b8-5f22-418d-9f9a-dba7549ade07:435'})]

In [9]:
from neo4j_graphrag.retrievers import VectorRetriever

vector_retriever = VectorRetriever(
   neo4j_driver,
   index_name=INDEX_NAME,
   embedder=embedder,
   return_properties=["text"],
)

In [10]:
import json

query_text = "When will Tesla's first quarter 2023 financial results conference call be webcast, and where can it be accessed? How long will the replay be available?"
vector_res = vector_retriever.get_search_results(query_text = query_text,
             top_k=3)
for i in vector_res.records: print("====\n" + json.dumps(i.data(), indent=4))


====
{
    "node": {
        "text": " I O N\nWEBCAST INFORMATION\nTesla will provide a live webcast of its first quarter 2023 financial results conference call beginning at 4:30 p.m. CT on April 19, 2023 at ir.tesla.com. This webcast will also be available for replay for approximately one year \nthereafter.\nCERTAIN TERMS\nWhen used in this update, certain terms have the following meanings. Our vehicle deliveries include only vehicles that have been transferred to end customers with all paperwork correctly completed. Our energy product\n"
    },
    "nodeLabels": [
        "__KGBuilder__",
        "Chunk"
    ],
    "elementId": "4:91ec46b8-5f22-418d-9f9a-dba7549ade07:740",
    "id": "4:91ec46b8-5f22-418d-9f9a-dba7549ade07:740",
    "score": 0.6626668572425842
}
====
{
    "node": {
        "text": " 21 186 83 69 115 223 292 346 205 305 276 261 \nDepreciation, amortization and impairment 530 577 553 567 584 618 621 681 761 848 880 922 956 989 1,046 \nStock-based compensation expense 1

In [11]:
from neo4j_graphrag.retrievers import VectorCypherRetriever

vc_retriever = VectorCypherRetriever(
   neo4j_driver,
   index_name=INDEX_NAME,
   embedder=embedder,
   retrieval_query="""
//1) Go out 2-3 hops in the entity graph and get relationships
WITH node AS chunk
MATCH (chunk)<-[:FROM_CHUNK]-()-[relList:!FROM_CHUNK]-{1,2}()
UNWIND relList AS rel

//2) collect relationships and text chunks
WITH collect(DISTINCT chunk) AS chunks,
 collect(DISTINCT rel) AS rels

//3) format and return context
RETURN '=== text ===\n' + apoc.text.join([c in chunks | c.text], '\n---\n') + '\n\n=== kg_rels ===\n' +
 apoc.text.join([r in rels | startNode(r).name + ' - ' + type(r) + '(' + coalesce(r.details, '') + ')' +  ' -> ' + endNode(r).name ], '\n---\n') AS info
"""
)

In [13]:
vc_res = vc_retriever.get_search_results(query_text = query_text, top_k=1)

# print output
kg_rel_pos = vc_res.records[0]['info'].find('\n\n=== kg_rels ===\n')
print("# Text Chunk Context:")
print(vc_res.records[0]['info'][:kg_rel_pos])
print("# KG Context From Relationships:")
print(vc_res.records[0]['info'][kg_rel_pos:])

# Text Chunk Context:
=== text ===
 Class C shares. The repurchases are expected to be executed from time to 
time, subject to general business and market conditions and other investment opportunities, through open market 
purchases or privately negotiated transactions, including through Rule 10b5-1 plans.
Webcast and conference call information
A live audio webcast of our first quarter  2023 earnings release call will be available on YouTube at https://
www.youtube.com/watch?v=76CVRgZUfps. The call begins today at 2:00 PM (PT) /
# KG Context From Relationships:


=== kg_rels ===
Unnamed Company - reports(Earnings release call for first quarter 2023) -> First Quarter 2023
---
Meta - reports(Meta reports First Quarter 2023 results) -> First Quarter 2023
---
Meta - includes(Meta will host a conference call to discuss the results at 2 p.m. PT / 5 p.m. ET today.) -> Webcast and Conference Call Information
---
Meta - includes(Meta will host a conference call to discuss the results) -> earni

In [14]:
from neo4j_graphrag.generation import RagTemplate
from neo4j_graphrag.generation.graphrag import GraphRAG

rag_template = RagTemplate(template='''Answer the Question using the following Context. Only respond with information mentioned in the Context. Do not inject any speculative information not mentioned.

# Question:
{query_text}

# Context:
{context}

# Answer:
''', expected_inputs=['query_text', 'context'])

v_rag  = GraphRAG(llm=llm, retriever=vector_retriever, prompt_template=rag_template)
vc_rag = GraphRAG(llm=llm, retriever=vc_retriever, prompt_template=rag_template)

In [18]:
query_text = "Tell me the relation between Meta Platforms, Inc. and financial results"
k=3
print(f"Vector Response: \n{v_rag.search(query_text=query_text, retriever_config={'top_k':k}).answer}")
print("\n===========================\n")
print(f"Vector + Cypher Response: \n{vc_rag.search(query_text=query_text, retriever_config={'top_k':k}).answer}")

Vector Response: 
The provided context does not mention any information about Meta Platforms, Inc. and its financial results.


Vector + Cypher Response: 
Meta Platforms, Inc. reports financial results for the first quarter of 2023.


In [23]:
query_text = "What is the Total liabilities of Meta?"
k=5
print(f"Vector Response: \n{v_rag.search(query_text=query_text, retriever_config={'top_k':k}).answer}")
print("\n===========================\n")
print(f"Vector + Cypher Response: \n{vc_rag.search(query_text=query_text, retriever_config={'top_k':k}).answer}")

Vector Response: 
The total liabilities of Meta are $59,696 million.


Vector + Cypher Response: 
Total liabilities 59,696 60,014


In [41]:
from neo4j_graphrag.generation import RagTemplate
from neo4j_graphrag.generation.graphrag import GraphRAG

rag_template = RagTemplate(template='''Answer the Question using the following Context. Only respond with information mentioned in the Context. Do not inject any speculative information not mentioned.
if the context has table and its relevant to the question, 
then use the table to answer the question in markdown format.
Always provide a useful insigh after the answer, and mention the source(document/file/chunk) of the information.
Never say that your answer is based on the context,

# Question:
{query_text}

# Context:
{context}

# Answer:
''', expected_inputs=['query_text', 'context'])

v_rag  = GraphRAG(llm=llm, retriever=vector_retriever, prompt_template=rag_template)
vc_rag = GraphRAG(llm=llm, retriever=vc_retriever, prompt_template=rag_template)

query_text = "Tell me about Nvidia free cash flow and its partners’ products, and then tell me about Meta Platforms, Inc. with its financial results"
k=1
display(Markdown(f"Vector Response: \n{v_rag.search(query_text=query_text, retriever_config={'top_k':k}).answer}"))
print("\n===========================\n")
display(Markdown(f"Vector + Cypher Response: \n{vc_rag.search(query_text=query_text, retriever_config={'top_k':k}).answer}"))

Vector Response: 
The context provided does not contain specific information about Nvidia's free cash flow, its partners' products, or Meta Platforms, Inc. and its financial results. Therefore, I cannot provide an answer based on the given context.

Vector + Cypher Response: 
Nvidia generates free cash flow and has partners' products. 

Meta Platforms, Inc. reported financial results for the third quarter of 2023, with revenue of $34.15 billion, up 23% year-over-year, and a net income of $11.58 billion, up 164% year-over-year.

**Insight:** Nvidia's financial performance includes generating free cash flow and collaborating with partners on products, while Meta Platforms, Inc. has shown significant growth in both revenue and net income in the third quarter of 2023.

**Source:** Document

In [44]:
k = 2
v_rag_result = v_rag.search(query_text=query_text, retriever_config={'top_k': k}, return_context=True)
vc_rag_result = vc_rag.search(query_text=query_text, retriever_config={'top_k': k}, return_context=True)

display(Markdown(f"Vector Response: \n{v_rag_result.answer}"))
print("\n===========================\n")
display(Markdown(f"Vector + Cypher Response: \n{vc_rag_result.answer}"))

Vector Response: 
### Nvidia Free Cash Flow and Partners’ Products

Nvidia's revenue was a record $8.29 billion, up 46% from a year ago and up 8% sequentially, with record revenue achieved in Gaming and Data Center. Gaming revenue was up 31% from a year ago and up 6% sequentially. The year-on-year increase reflects higher sales of GeForce.

### Meta Platforms, Inc. Financial Results

#### Condensed Consolidated Statements of Cash Flows (In millions, Unaudited)

| Three Months Ended March 31 | 2023  | 2022  |
|-----------------------------|-------|-------|
| Net income                  | 5,709 | 7,465 |

**Insight:** Nvidia has shown significant growth in its revenue, particularly in the Gaming and Data Center segments. On the other hand, Meta Platforms, Inc. reported a net income of $5.709 billion for the three months ended March 31, 2023, which is a decrease from $7.465 billion in the same period in 2022.

**Source:** Document

Vector + Cypher Response: 
### Nvidia Free Cash Flow and Partners’ Products

Nvidia generates free cash flow and has various partners' products. The company addresses four large markets where its expertise is critical: Gaming, Data Center, Professional Visualization, and Automotive. Nvidia's revenue was a record $8.29 billion, up 46% from a year ago and up 8% sequentially, with record revenue achieved in Gaming and Data Center.

### Meta Platforms, Inc. Financial Results

Meta Platforms, Inc. reported the following financial results for the three months ended March 31, 2023:

| Financial Metric | Q1 2023 | Q1 2022 |
|------------------|---------|---------|
| Net Income       | $5,709 million | $7,465 million |
| Cash Flows from Operating Activities | - | - |

Meta Platforms, Inc. experienced a decrease in net income from $7,465 million in Q1 2022 to $5,709 million in Q1 2023.

**Insight:** Nvidia continues to show strong growth in its key markets, particularly in Gaming and Data Center, while Meta Platforms, Inc. has seen a decline in net income year-over-year.

**Source:** Document provided.

In [45]:
for i in v_rag_result.retriever_result.items: 
    print(json.dumps(eval(i.content), indent=1))

{
 "text": " interconnects, software, algorithms, systems, and \nservices to deliver unique value. Our platforms address four large markets where our expertise is critical: \nGaming, Data Center, Professional Visualization, and Automotive.\nRevenue\nRevenue was a record $8.29 billion, up 46% from a year ago and up 8% sequentially, with record revenue \nachieved in Gaming and Data Center. \nGaming revenue was up 31% from a year ago and up 6% sequentially. The year-on-year increase reflects \nhigher sales of GeForce"
}
{
 "text": ",241 64,799\nTotal stockholders' equity 124,795 125,713\nTotal liabilities and stockholders' equity\n$\u00a0\u00a0\u00a0\u00a0\u00a0\u00a0\u00a0\u00a0\u00a0\u00a0\u00a0\u00a0\u00a0\u00a0\u00a0184,491 $\u00a0\u00a0\u00a0\u00a0\u00a0\u00a0\u00a0\u00a0\u00a0\u00a0\u00a0\u00a0\u00a0\u00a0\u00a0 185,727\n \n \n \nMETA PLATFORMS, INC.\nCONDENSED CONSOLIDATED STATEMENTS OF CASH FLOWS\n(In millions)\n(Unaudited)\n Three Months Ended March 31,\n 2023 2022\nCash \u0000ow

In [50]:
vc_ls = vc_rag_result.retriever_result.items[0].content.split('\\n---\\n')
print("\n".join(i for i in vc_ls if "meta" in i.lower() or "nvidia" in i.lower()))

,241 64,799\nTotal stockholders' equity 124,795 125,713\nTotal liabilities and stockholders' equity\n$\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0184,491 $\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0 185,727\n \n \n \nMETA PLATFORMS, INC.\nCONDENSED CONSOLIDATED STATEMENTS OF CASH FLOWS\n(In millions)\n(Unaudited)\n Three Months Ended March 31,\n 2023 2022\nCash \x00ows from operating activities    \nNet income $\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa05,709 $\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0 7,465\nAdjustments to reconcile net income to net cash provided by operating activities:   \nDepreciation and \n\n=== kg_rels ===\ninterconnects - includes(interconnects, software, algorithms, systems, and services to deliver unique value) -> services
NVIDIA - generates(NVIDIA generates Data Center revenue) -> Data Center revenue
NVIDIA - reports(reports GAAP financial measures) -> GAAP financial measures